# Lesson 15 — Deploying LangGraph Agents with FastAPI

## What you will learn
- Wrap a LangGraph agent as a **REST API** with FastAPI
- Handle **HITL interrupts** over HTTP (approve/reject endpoints)
- **Multi-user isolation** via `thread_id`
- **Streaming responses** via Server-Sent Events (SSE)
- Production run command and **Dockerfile**

## Architecture
```
Client (browser / curl)
    ↓  POST /ask  {question, user_id}
FastAPI app
    ↓  graph.invoke(state, config={thread_id: user_id})
LangGraph agent  (SqliteSaver persists state)
    ↓  if interrupt needed
    ↑  POST /approve  {thread_id, decision}
Client
```

## Run command
```bash
uvicorn lesson_15_deployment.api:app --host 0.0.0.0 --port 8000 --reload
```

In [ ]:
# !pip install fastapi uvicorn python-dotenv langgraph langchain-ollama

## Step 1 — The Core Agent (same as Lesson 10 capstone)

We keep the agent logic unchanged — only wrap it in HTTP.

In [ ]:
import sqlite3, os
from typing import Annotated
from typing_extensions import TypedDict
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

llm = ChatOllama(model="llama3.2", temperature=0)

@tool
def get_time() -> str:
    """Get the current server time."""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

tools = [get_time, add]
llm_with_tools = llm.bind_tools(tools)

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

def agent_node(state: AgentState) -> dict:
    response = llm_with_tools.invoke([SystemMessage(content="You are a helpful assistant.")] + state["messages"])
    return {"messages": [response]}

def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    return "tools" if (hasattr(last, "tool_calls") and last.tool_calls) else "end"

checkpointer = MemorySaver()
builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
builder.add_edge("tools", "agent")
graph = builder.compile(checkpointer=checkpointer)

print("Agent compiled! Ready to wrap in FastAPI.")

## Step 2 — FastAPI Wrapper (conceptual walkthrough)

The actual server is in `api.py`. Here we show the key patterns.

In [ ]:
# This is the STRUCTURE of api.py — shown here for learning.
# Run the actual server with: uvicorn lesson_15_deployment.api:app --port 8000

FASTAPI_STRUCTURE = """
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="LangGraph Agent API")

class AskRequest(BaseModel):
    question: str
    user_id:  str

class AskResponse(BaseModel):
    answer:     str
    thread_id:  str
    interrupted: bool

@app.post("/ask", response_model=AskResponse)
async def ask(req: AskRequest):
    config = {"configurable": {"thread_id": req.user_id}}
    result = graph.invoke(
        {"messages": [HumanMessage(content=req.question)]},
        config=config
    )
    answer = result["messages"][-1].content
    return AskResponse(answer=answer, thread_id=req.user_id, interrupted=False)

@app.get("/health")
async def health():
    return {"status": "ok"}
"""
print(FASTAPI_STRUCTURE)

## Step 3 — Multi-User Isolation

Each `thread_id` is an isolated conversation. Different users never see each other's state.

In [ ]:
def simulate_api_call(user_id: str, question: str):
    config = {"configurable": {"thread_id": user_id}}
    result = graph.invoke(
        {"messages": [HumanMessage(content=question)]},
        config=config
    )
    return result["messages"][-1].content

# Two users in parallel — fully isolated
print("=== User alice ===")
print(simulate_api_call("alice", "What is 5 + 3?"))

print("\n=== User bob ===")
print(simulate_api_call("bob", "What time is it?"))

print("\n=== alice follow-up (remembers context) ===")
print(simulate_api_call("alice", "Now multiply that result by 10"))

## Step 4 — Streaming with SSE

`graph.stream()` yields state updates node-by-node — perfect for SSE endpoints.

In [ ]:
config = {"configurable": {"thread_id": "stream-demo"}}

print("Streaming node updates:")
for chunk in graph.stream(
    {"messages": [HumanMessage(content="What is 12 + 8?")]},
    config=config,
    stream_mode="updates",
):
    for node_name, update in chunk.items():
        if "messages" in update and update["messages"]:
            last = update["messages"][-1]
            content = getattr(last, "content", "")
            if content:
                print(f"  [{node_name}]: {content[:100]}")

## Step 5 — Dockerfile Overview

The `Dockerfile` in this lesson folder is ready to use.

In [ ]:
dockerfile_content = open("Dockerfile").read() if os.path.exists("Dockerfile") else "(run from lesson_15_deployment directory)"
print(dockerfile_content)

## Key Takeaways

| Pattern | Code |
|---------|------|
| Per-user isolation | `config = {"configurable": {"thread_id": user_id}}` |
| Invoke over HTTP | `graph.invoke(state, config)` in FastAPI handler |
| Streaming SSE | `graph.stream(..., stream_mode="updates")` |
| Persist across restarts | Replace `MemorySaver` with `SqliteSaver` |
| Container deploy | `uvicorn app:app --host 0.0.0.0 --port 8000 --workers 4` |

## 🏋️ Exercise
1. Start the actual API: `uvicorn lesson_15_deployment.api:app --reload --port 8000`
2. Test with curl:
   ```bash
   curl -X POST http://localhost:8000/ask \
        -H 'Content-Type: application/json' \
        -d '{"question": "What is 25 + 17?", "user_id": "test-user"}'
   ```
3. Add a `GET /history/{user_id}` endpoint that returns the full message history for a user
4. Add rate limiting: reject requests if the same user sends > 10 requests per minute

In [ ]:
# Your exercise solution here
